# FinMentor Backend — Complete Walkthrough

This notebook explains every file in the backend, what it does, why it's designed that way, how to learn each concept yourself — and documents every bug found during code review with exact fixes.

**Reading order:** Go cell by cell. Each section builds on the previous one.

---

## The big picture first

When a user sends a message from your React app, here's what happens in the backend:

```
User message
     ↓
main.py receives it via POST /chat
     ↓
market_data.py fetches live NSE/RBI/Forex data
     ↓
financial_engine.py runs all the math (SIP, FIRE, insurance gap...)
     ↓
behaviour_profiler.py detects which persona the user is
     ↓
system_prompt.py assembles all of the above into one big instruction string
     ↓
Gemini API gets the instruction + user message → generates a response
     ↓
Response sent back to React frontend as JSON
```

Each of these steps is a separate Python file. Let's go through them one by one.

---

## ⚠️ Part 0 — Bug Report (Read This First)

A full review of all backend files found **8 bugs** — ranging from silent validation failures to a logic error in the FIRE simulation. Each is documented below with the exact location, the problem, and the fix.

---

### Bug 1 — CRITICAL: `min_items` / `max_items` silently ignored in Pydantic v2
**File:** `main.py`, lines 107–110  
**Severity:** 🔴 Critical — validation never runs, malformed requests accepted

```python
# ❌ BROKEN — these are Pydantic v1 constraint names. Pydantic v2 silently ignores them.
quiz_answers: list[str] = Field(
    ...,
    min_items=3,   # <-- silently ignored
    max_items=3,   # <-- silently ignored
)

# ✅ FIX — Pydantic v2 uses min_length / max_length for sequence fields
quiz_answers: list[str] = Field(
    ...,
    min_length=3,
    max_length=3,
)
```

**Why it matters:** Without the fix, a user can send 0 or 10 quiz answers and the server will happily accept it. `detect_persona()` will silently use empty defaults for missing answers, producing a wrong persona. The user gets bad advice and you get no error to debug.

---

### Bug 2 — MODERATE: `@validator` is Pydantic v1 syntax
**File:** `main.py`, lines 16, 98–101  
**Severity:** 🟠 Moderate — works now via compatibility shim, will break in future Pydantic releases

```python
# ❌ BROKEN — @validator is deprecated in Pydantic v2
from pydantic import BaseModel, Field, validator

@validator("monthly_expenses")
def expenses_less_than_income(cls, v, values):
    return v

# ✅ FIX — use @field_validator with @classmethod in Pydantic v2
from pydantic import BaseModel, Field, field_validator

@field_validator("monthly_expenses")
@classmethod
def expenses_less_than_income(cls, v, info):
    # info.data holds previously validated fields
    income = info.data.get("monthly_income")
    if income and v > income:
        # Could log a warning here — expenses exceed income
        pass
    return v
```

**Why it matters:** Pydantic v2 is stricter. The old `@validator` decorator will raise a `PydanticUserError` in future minor releases. The API for accessing sibling fields also changed (`values` → `info.data`).

---

### Bug 3 — MODERATE: Inconsistent defaults in `cpi_inflation` ternary
**File:** `main.py`, lines 148–150  
**Severity:** 🟠 Moderate — wrong inflation rate used if the market API key is ever missing

```python
# ❌ BROKEN — three separate .get() calls with DIFFERENT defaults
inflation=market.get("cpi_inflation", 0.06) / 100   # default: 0.06 (decimal!)
if market.get("cpi_inflation", 6.0) > 1              # default: 6.0  (percent!)
else market.get("cpi_inflation", 0.06),              # default: 0.06 (decimal!)

# If key is missing:
#   → first .get() returns 0.06
#   → second .get() returns 6.0, so condition is True
#   → result = 0.06 / 100 = 0.0006  ← completely wrong! (0.06% inflation)

# ✅ FIX — read once, convert once
raw_inflation = market.get("cpi_inflation", 6.0)   # always stored as percent (5.1)
inflation = raw_inflation / 100 if raw_inflation > 1 else raw_inflation
# Then pass: inflation=inflation
```

**Why it matters:** With the bug, a missing `cpi_inflation` key feeds `0.0006` (0.06%) to `calc_fire_date`. The FIRE corpus target becomes absurdly small and FIRE age will be calculated as almost immediate. A user could see "You can retire in 2 years!" during a demo if the market API had a hiccup.

---

### Bug 4 — LOGIC: `monthly_savings` never updates as expenses inflate
**File:** `financial_engine.py`, function `calc_fire_date`, lines 232–241  
**Severity:** 🟠 Moderate — FIRE date consistently shown earlier than it actually is

```python
# ❌ BROKEN — savings computed ONCE before the loop
monthly_savings = monthly_income - monthly_expense  # e.g. ₹28,000

for year in range(1, max_years + 1):
    corpus = corpus * (1 + annual_return)
    corpus += monthly_savings * 12      # Always adds ₹28,000 × 12
    monthly_expense *= (1 + inflation)  # ← expense grows each year
    # But monthly_savings still uses the OLD expense number!

# ✅ FIX — recalculate savings inside the loop after inflating expenses
for year in range(1, max_years + 1):
    corpus = corpus * (1 + annual_return)
    corpus += monthly_savings * 12         # Use this year's savings
    monthly_expense *= (1 + inflation)     # Grow expenses for next year
    monthly_income *= (1 + salary_growth)  # Also grow income (add param, default 0.08)
    monthly_savings = monthly_income - monthly_expense  # Recalculate for next year
    fire_corpus_target = calc_fire_corpus(monthly_expense, inflation)
```

**Why it matters:** In year 1, surplus might be ₹28,000/month. By year 10, expenses have grown 80% due to inflation — real surplus shrinks significantly. The broken code keeps adding ₹28K forever, making the portfolio look far larger than it will actually be. FIRE date can appear 5–8 years earlier than the truth.

---

### Bug 5 — MINOR: `user.dict()` deprecated in Pydantic v2
**File:** `main.py`, line 259  
**Severity:** 🟡 Minor — works today, will break in future Pydantic releases

```python
# ❌ BROKEN — .dict() is Pydantic v1
system = build_system_prompt(
    user=user.dict(),   # Raises DeprecationWarning in Pydantic v2
    ...
)

# ✅ FIX — use .model_dump() in Pydantic v2
system = build_system_prompt(
    user=user.model_dump(),
    ...
)
```

---

### Bug 6 — MINOR: `import asyncio` inside a function body
**File:** `market_data.py`, inside `get_market_snapshot()`  
**Severity:** 🟡 Minor — works but violates Python convention, harms readability

```python
# ❌ BROKEN — import buried inside a function
async def get_market_snapshot() -> dict:
    import asyncio          # ← This works but is wrong style
    usd_inr, gold_price, nifty_data = await asyncio.gather(...)

# ✅ FIX — put ALL imports at the top of the file
import asyncio             # ← top of market_data.py, with the other imports
import os
import httpx
...
```

---

### Bug 7 — MINOR: No validation that quiz answer strings are valid keys
**File:** `main.py` / `behaviour_profiler.py`  
**Severity:** 🟡 Minor — invalid answers silently produce wrong persona

```python
# ❌ PROBLEM — if frontend sends a typo like "invest-all" instead of "invest_all",
# detect_persona() gives zero points for Q1, silently continues, and returns
# a wrong persona with no error raised.

# ✅ FIX — validate against known keys in main.py
from behaviour_profiler import (
    Q1_INVEST_ALL, Q1_INVEST_HALF, Q1_PAY_DEBT, Q1_BUY_WANT, Q1_SAVE_FAMILY,
    Q2_NOT_ENOUGH, Q2_MISSING_OUT, Q2_DEBT_TRAP, Q2_FAMILY_BURDEN, Q2_MARKET_CRASH,
    Q3_BEFORE_45, Q3_AT_60, Q3_NEVER, Q3_FLEXIBLE,
)

VALID_Q1 = {Q1_INVEST_ALL, Q1_INVEST_HALF, Q1_PAY_DEBT, Q1_BUY_WANT, Q1_SAVE_FAMILY}
VALID_Q2 = {Q2_NOT_ENOUGH, Q2_MISSING_OUT, Q2_DEBT_TRAP, Q2_FAMILY_BURDEN, Q2_MARKET_CRASH}
VALID_Q3 = {Q3_BEFORE_45, Q3_AT_60, Q3_NEVER, Q3_FLEXIBLE}

@field_validator("quiz_answers")
@classmethod
def validate_quiz_keys(cls, v):
    valid_sets = [VALID_Q1, VALID_Q2, VALID_Q3]
    for i, (answer, valid) in enumerate(zip(v, valid_sets)):
        if answer not in valid:
            raise ValueError(f"Q{i+1} answer '{answer}' is not a valid key. Valid: {valid}")
    return v
```

---

### Bug 8 — MINOR: `Balanced Builder` persona missing from `persona_map`
**File:** `behaviour_profiler.py`  
**Severity:** 🟡 Minor — "Balanced Builder" is listed in the header comment as a 6th persona, but never implemented

```python
# ❌ PROBLEM — file header mentions 6 personas including "Balanced Builder"
# but persona_map only has 5 keys: fire, yolo, family, debt, wealth
# If a user scores equally on multiple traits, the winner defaults to
# whichever key Python's max() picks first — not an explicit "balanced" result.

# ✅ FIX — either remove "Balanced Builder" from the comment,
# or implement it and give it a score path:

# 1. After computing winner, check if scores are too close:
sorted_scores = sorted(scores.values(), reverse=True)
if sorted_scores[0] - sorted_scores[1] <= 2:   # Top two are within 2 points
    winner = "balanced"                          # Too close to call → balanced

# 2. Add to persona_map:
"balanced": Persona(
    type="Balanced Builder",
    tone="friendly",
    framing="This user has no single dominant financial priority...",
    risk_profile="moderate",
    risk_score=risk_score,
    primary_motivation="Steady, sustainable financial progress",
    primary_fear="Making the wrong financial move",
    plan_style="balanced",
    emoji_label="Balanced wealth building",
)
```

---

### Summary Table

| # | File | Line(s) | Severity | Impact |
|---|------|---------|----------|--------|
| 1 | main.py | 107–110 | 🔴 Critical | Quiz validation bypassed entirely |
| 2 | main.py | 16, 98–101 | 🟠 Moderate | Deprecated validator, will break |
| 3 | main.py | 148–150 | 🟠 Moderate | Wrong inflation = wrong FIRE date |
| 4 | financial_engine.py | 232–241 | 🟠 Moderate | FIRE date 5–8 years too optimistic |
| 5 | main.py | 259 | 🟡 Minor | Deprecated .dict() |
| 6 | market_data.py | inside fn | 🟡 Minor | Import in function body |
| 7 | main.py / profiler | — | 🟡 Minor | Invalid quiz keys accepted silently |
| 8 | behaviour_profiler.py | header | 🟡 Minor | 6th persona promised, not delivered |

---
## Part 1 — FastAPI basics (`main.py`)

### What is FastAPI?

FastAPI is a Python web framework. A **web framework** lets you write Python functions and turn them into API endpoints — URLs that your React app can call.

Without FastAPI, your Python code has no way to receive HTTP requests from a browser.

### The minimum viable FastAPI app

In [ ]:
# pip install fastapi uvicorn

from fastapi import FastAPI

app = FastAPI()

@app.get("/hello")
def say_hello():
    return {"message": "Hello from Python!"}

# Run this with: uvicorn main:app --reload
# Then open: http://localhost:8000/hello
# You'll see: {"message": "Hello from Python!"}

print("This cell is just for reference — run it via uvicorn, not Jupyter")

### Pydantic models — how FastAPI validates data

When your React app sends user data (age, income, etc.), FastAPI uses **Pydantic** to validate it automatically. If the user sends a string where an integer is expected, FastAPI rejects it with a clear error — before your code even runs.

**Important:** The project uses Pydantic **v2** (`pydantic==2.7.1` in requirements.txt). Several Pydantic v1 APIs were used in the original code and are now fixed below.

In [ ]:
# ── Pydantic v2 correct patterns (fixes Bugs 1, 2, 5) ─────────
from pydantic import BaseModel, Field, field_validator
from typing import Optional

class UserProfile(BaseModel):
    age: int = Field(..., ge=18, le=70)          # ge/le still the same in v2
    monthly_income: float = Field(..., gt=0)
    monthly_expenses: float = Field(..., gt=0)
    name: str = "Friend"
    goals: Optional[list[str]] = ["retirement"]

    # ✅ Correct Pydantic v2 validator syntax
    @field_validator("monthly_expenses")
    @classmethod
    def expenses_less_than_income(cls, v, info):
        income = info.data.get("monthly_income")  # info.data in v2 (not values)
        if income and v > income:
            print(f"Warning: expenses ({v}) exceed income ({income})")
        return v

class ChatRequest(BaseModel):
    user: UserProfile
    # ✅ Correct Pydantic v2 list length validation (fixes Bug 1)
    quiz_answers: list[str] = Field(..., min_length=3, max_length=3)
    message: str = Field(..., min_length=1, max_length=2000)

# Test valid input
try:
    u = UserProfile(age=25, monthly_income=60000, monthly_expenses=35000)
    print("Valid user:", u)
except Exception as e:
    print("Error:", e)

# Test invalid age
try:
    u2 = UserProfile(age=15, monthly_income=60000, monthly_expenses=35000)
except Exception as e:
    print("Validation caught:", e)

# ✅ Correct way to serialize to dict in Pydantic v2 (fixes Bug 5)
u = UserProfile(age=28, monthly_income=75000, monthly_expenses=50000)
print("model_dump():", u.model_dump())

### async/await — why the functions look different

You'll notice some functions in `main.py` have `async def` instead of just `def`. This is Python's **asynchronous programming** — it lets the server handle multiple requests at the same time without waiting.

The key rule: if a function calls an API or does I/O (file read, network call), make it `async`. Otherwise, use regular `def`.

In [ ]:
import asyncio
import time

# SLOW way — sequential, each waits for the previous
def slow_fetch_all():
    start = time.time()
    time.sleep(1)  # Pretend: fetch NSE data (1 second)
    time.sleep(1)  # Pretend: fetch forex data (1 second)
    time.sleep(1)  # Pretend: fetch gold data (1 second)
    print(f"Sequential: {time.time() - start:.1f}s")

# FAST way — concurrent, all run at the same time
async def fast_fetch_all():
    start = time.time()
    await asyncio.gather(
        asyncio.sleep(1),  # Pretend: fetch NSE data
        asyncio.sleep(1),  # Pretend: fetch forex data
        asyncio.sleep(1),  # Pretend: fetch gold data
    )
    print(f"Concurrent: {time.time() - start:.1f}s")

slow_fetch_all()           # ~3 seconds
await fast_fetch_all()     # ~1 second — 3x faster!

### CORS — why your React app can't talk to Python without it

CORS (Cross-Origin Resource Sharing) is a browser security rule. By default, a browser **blocks** a webpage from calling an API on a different domain/port. Your React app on `localhost:5173` can't call your Python server on `localhost:8000` without CORS being enabled.

This is why `main.py` includes the `CORSMiddleware` setup. Without it, every request from React to Python will fail with a CORS error in the browser console.

In [ ]:
# CORS setup in main.py (pattern reference, not directly runnable)
from fastapi.middleware.cors import CORSMiddleware

# app.add_middleware(
#     CORSMiddleware,
#     allow_origins=[
#         "http://localhost:5173",        # Your React dev server
#         "https://your-app.vercel.app",  # Your production frontend
#     ],
#     allow_methods=["*"],   # Allow GET, POST, PUT, DELETE, etc.
#     allow_headers=["*"],   # Allow all headers
# )

# ── Common CORS mistake to watch out for ──────────────────────
# If you have allow_credentials=True AND allow_origins=["*"]
# the browser will reject it with a CORS error anyway.
# Rule: use credentials=True only with SPECIFIC origins, never with wildcard.

print("CORS allows your React app to call this Python server")
print("Without it: browser blocks all requests with a CORS error")
print("Never use allow_origins=['*'] with allow_credentials=True")

### The `/chat` endpoint — complete corrected flow

In [ ]:
# This is the corrected version of run_calculations and the /chat handler
# applying all bug fixes. Reference only — requires FastAPI running context.

def run_calculations_fixed(user, nifty_pe, market):
    """
    Fixed version of run_calculations.
    Fixes Bug 3: consistent inflation handling.
    """
    # ✅ Fix Bug 3: read once, convert once
    raw_inflation = market.get("cpi_inflation", 6.0)  # stored as percent e.g. 5.1
    inflation = raw_inflation / 100 if raw_inflation > 1 else raw_inflation

    fire = calc_fire_date_fixed(
        current_age=user.age,
        monthly_income=user.monthly_income,
        monthly_expense=user.monthly_expenses,
        current_corpus=user.existing_investments,
        annual_return=0.12,
        inflation=inflation,   # ← now always a decimal like 0.051
    )
    return fire

def calc_fire_date_fixed(
    current_age, monthly_income, monthly_expense,
    current_corpus, annual_return=0.12, inflation=0.06,
    salary_growth=0.08,  # ← new param: typical Indian salary growth ~8%
):
    """
    Fixed version — fixes Bug 4: savings recalculated each year.
    Added salary_growth parameter for realistic modelling.
    """
    fire_corpus_target = monthly_expense * 12 * (1 + inflation) * 25
    monthly_savings = monthly_income - monthly_expense
    corpus = current_corpus
    age = current_age

    for year in range(1, 51):
        corpus = corpus * (1 + annual_return)      # Grow existing corpus
        corpus += monthly_savings * 12             # Add this year's savings

        # ✅ Now update both income and expense BEFORE recalculating savings
        monthly_income  *= (1 + salary_growth)     # Salary grows ~8%/yr
        monthly_expense *= (1 + inflation)          # Expenses grow ~6%/yr
        monthly_savings  = monthly_income - monthly_expense  # Recalculate!

        fire_corpus_target = monthly_expense * 12 * (1 + inflation) * 25
        age += 1

        if corpus >= fire_corpus_target:
            return {
                "fire_age": age,
                "years_to_fire": year,
                "fire_corpus_target": round(fire_corpus_target, 2),
                "projected_corpus": round(corpus, 2),
                "achievable": True,
            }

    return {"fire_age": None, "years_to_fire": None, "achievable": False}

# ── Compare buggy vs fixed for a real user ────────────────────
def buggy_fire(income, expense, corpus, age):
    """Original broken version."""
    monthly_savings = income - expense
    for year in range(1, 51):
        corpus = corpus * 1.12
        corpus += monthly_savings * 12        # ← never updates
        expense *= 1.06
        if corpus >= expense * 12 * 1.06 * 25:
            return age + year
    return None

def fixed_fire(income, expense, corpus, age):
    """Fixed version."""
    savings = income - expense
    for year in range(1, 51):
        corpus = corpus * 1.12
        corpus += savings * 12
        income  *= 1.08    # salary grows
        expense *= 1.06    # expenses grow
        savings  = income - expense
        if corpus >= expense * 12 * 1.06 * 25:
            return age + year
    return None

# Arjun: 27 years old, ₹80K income, ₹52K expenses, ₹3L corpus
buggy_result = buggy_fire(80000, 52000, 300000, 27)
fixed_result = fixed_fire(80000, 52000, 300000, 27)
print(f"Buggy FIRE age:  {buggy_result}  (overoptimistic)")
print(f"Fixed FIRE age:  {fixed_result}  (realistic)")
print(f"Difference:      {fixed_result - buggy_result} years!")

### 📚 Resources to learn FastAPI

- **Official FastAPI docs** — `fastapi.tiangolo.com` — the best framework documentation in Python. Read 'First Steps' and 'Request Body' sections first.
- **FastAPI full course** — Search 'FastAPI tutorial freeCodeCamp' on YouTube — 20-minute crash course, covers everything you need.
- **Pydantic v2 migration guide** — `docs.pydantic.dev/latest/migration/` — Critical reading since this project uses v2. Shows every API that changed from v1.
- **Pydantic v2 validators** — `docs.pydantic.dev/latest/concepts/validators/` — The new `@field_validator` and `@model_validator` docs.
- **Async Python** — Search 'Python async await explained simply' on Real Python (`realpython.com`) — excellent visual explanations.
- **Postman** — `postman.com` — Test your API endpoints without needing the React frontend. Essential during development.

---
## Part 2 — Financial Math (`financial_engine.py`)

This file has zero AI and zero API calls. Just math. Let's understand each formula and the bug that existed.

In [ ]:
# ── SIP Formula ───────────────────────────────────────────────
# SIP = FV × r / [(1+r)^n − 1]
# FV = target amount, r = monthly return, n = total months
#
# This answers: "How much should I invest every month to reach ₹X in Y years?"

def calc_sip(target, years, annual_rate=0.12):
    n = years * 12           # total months
    r = annual_rate / 12     # monthly rate (12% yearly = 1% monthly)
    sip = target * r / ((1 + r) ** n - 1)
    return round(sip, 2)

# Example: To have ₹1 crore in 20 years at 12% CAGR
monthly_sip = calc_sip(target=10_000_000, years=20, annual_rate=0.12)
print(f"Monthly SIP needed for ₹1 crore in 20 years: ₹{monthly_sip:,}")

# The power of starting early — compare starting at 25 vs 35
sip_25 = calc_sip(10_000_000, years=35)
sip_35 = calc_sip(10_000_000, years=25)
print(f"\nStarting at 25 (35 years): ₹{sip_25:,}/month")
print(f"Starting at 35 (25 years): ₹{sip_35:,}/month")
print(f"10-year delay costs:         ₹{sip_35 - sip_25:,}/month extra")

In [ ]:
# ── FIRE Corpus Formula ───────────────────────────────────────
# The 4% safe withdrawal rate means:
# If your corpus = 25x your annual expenses, you can withdraw
# 4% per year indefinitely (assuming ~8% growth, ~4% inflation).
#
# This is from the Trinity Study (1998) — a landmark personal finance paper.
# In India, we adjust for higher inflation (6-7% vs 3% in US).

def calc_fire_corpus(monthly_expense, inflation=0.065):
    annual_expense = monthly_expense * 12
    adjusted = annual_expense * (1 + inflation)  # Inflate by 1 year
    corpus = adjusted * 25                        # 25x multiplier = 4% SWR
    return round(corpus, 2)

# If monthly expenses = ₹50,000
corpus = calc_fire_corpus(50000)
print(f"FIRE corpus needed: ₹{corpus/100_000:.1f} lakh")

# ── Why India needs a bigger corpus than the US ───────────────
# In US: 4% withdrawal rate works because inflation is ~3%
# In India: inflation is 6-7%, so the corpus erodes faster
# Some Indian FIRE advocates use 3% withdrawal (33× expenses)

corpus_conservative = monthly_expense * 12 * 1.065 * 33  # 3% SWR
corpus_standard     = monthly_expense * 12 * 1.065 * 25  # 4% SWR
monthly_expense = 50000
corpus_conservative = monthly_expense * 12 * 1.065 * 33
print(f"\nConservative (3% SWR, 33x): ₹{corpus_conservative/100_000:.1f} lakh")
print(f"Standard     (4% SWR, 25x): ₹{corpus_standard/100_000:.1f} lakh")
print(f"Gap: ₹{(corpus_conservative - corpus_standard)/100_000:.1f} lakh — significant in India!")

In [ ]:
# ── Compound Interest (Future Value) ─────────────────────────
# FV = P × (1 + r)^n
# Answers: "What will my current ₹2 lakh in mutual funds be worth in 15 years?"

def calc_future_value(principal, annual_rate, years):
    return round(principal * (1 + annual_rate) ** years, 2)

# ₹2 lakh invested today at 12% CAGR for 15 years
fv = calc_future_value(200000, 0.12, 15)
print(f"₹2 lakh today → ₹{fv/100_000:.2f} lakh in 15 years at 12%")

# The Rule of 72 — quick mental math
rate = 12
doubles_in = 72 / rate
print(f"\nRule of 72: Money doubles in {doubles_in} years at {rate}%")
print(f"PPF at 7.1%: doubles in {72/7.1:.1f} years")
print(f"FD  at 7.2%: doubles in {72/7.2:.1f} years")
print(f"Nifty at 12%: doubles in {72/12:.1f} years")

In [ ]:
# ── Bug 4 Demo: FIRE date error caused by static savings ──────
# This directly shows the impact of the bug on a real user.

def compare_fire_simulations(income, expense, corpus, age, label="User"):
    print(f"\n{'='*55}")
    print(f"{label}: ₹{income/1000:.0f}K income, ₹{expense/1000:.0f}K expenses, Age {age}")
    print(f"{'='*55}")

    # Buggy version
    c, e, s, a = corpus, expense, income - expense, age
    for yr in range(1, 51):
        c = c * 1.12 + s * 12
        e *= 1.06         # expense grows
        # Bug: s never updates
        if c >= e * 12 * 1.065 * 25:
            print(f"  Buggy FIRE age:  {a + yr}  (surplus fixed at ₹{s/1000:.1f}K/month forever)")
            break

    # Fixed version
    c, e, i2, a = corpus, expense, income, age
    s2 = i2 - e
    for yr in range(1, 51):
        c = c * 1.12 + s2 * 12
        i2 *= 1.08        # income grows 8%
        e  *= 1.06        # expenses grow 6%
        s2  = i2 - e      # recalculate savings
        if c >= e * 12 * 1.065 * 25:
            print(f"  Fixed FIRE age:  {a + yr}  (surplus grows with salary)")
            break

compare_fire_simulations(80000, 52000, 300000, 27, "Arjun (median Indian IT professional)")
compare_fire_simulations(120000, 75000, 500000, 30, "Priya (senior engineer)")
compare_fire_simulations(50000, 40000, 100000, 24, "Rohan (fresh graduate, tight budget)")

In [ ]:
# ── Insurance Gap Calculator ──────────────────────────────────
# Term insurance target = 12× annual income (IRDAI guidance)
# Health cover target: ₹10L metro, ₹5L tier-2/3

def calc_insurance_gap(annual_income, existing_term, existing_health, city_tier="metro"):
    term_target   = annual_income * 12
    term_gap      = max(0, term_target - existing_term)
    health_target = 1_000_000 if city_tier == "metro" else 500_000
    health_gap    = max(0, health_target - existing_health)
    return {
        "term_target": term_target, "term_gap": term_gap,
        "health_target": health_target, "health_gap": health_gap,
    }

# Example: ₹8 lakh annual income, ₹50L existing term, ₹5L health
gap = calc_insurance_gap(800000, 5_000_000, 500_000, "metro")
print(f"Term target:  ₹{gap['term_target']/100_000:.0f}L  |  Gap: ₹{gap['term_gap']/100_000:.0f}L")
print(f"Health target: ₹{gap['health_target']/100_000:.0f}L  |  Gap: ₹{gap['health_gap']/100_000:.0f}L")
print()
# Key insight: why 12x income?
# If you earn ₹8L/yr and die, your family loses ₹8L/yr in income.
# ₹96L invested at 8% post-tax gives ₹7.7L/yr income — roughly replacing yours.
print("Why 12x? ₹96L at 8% yield = ₹7.7L/yr — replaces lost income permanently.")

### 📚 Resources to learn financial math for India

- **AMFI investor education** — `amfiindia.com/investor-education` — India-specific, SEBI-approved content. Free.
- **ET Money Learn** — `etmoney.com/learn` — India's best personal finance explainers. Read the SIP, FIRE, and asset allocation sections.
- **Monika Halan — Let's Talk Money** — The best Indian personal finance book. Available on Amazon for ₹200. Chapters 3, 5, 7 are directly relevant to this product.
- **SEBI Investor Charter** — `sebi.gov.in/investor-education` — Official 2-page charter. Gives your product regulatory credibility.
- **Trinity Study (for FIRE)** — Search 'Bengen 4 percent rule' — the original 1994 paper. Free PDF on ResearchGate.
- **Freefincal.com** — Best Indian FIRE blog. Search 'FIRE India freefincal' — detailed simulations adapted for India.
- **Investopedia SIP formula** — Search 'SIP formula Investopedia' — clear derivation with examples.

---
## Part 3 — Live Market Data (`market_data.py`)

This file fetches real-time data from free APIs. Understanding HTTP requests in Python is key here.

In [ ]:
# ── httpx is an async HTTP client ────────────────────────────
# Like 'requests', but works with async/await.
# Install: pip install httpx

import httpx

# Simple synchronous version (easier to understand)
def fetch_usd_inr_sync():
    """Fetches USD/INR rate — no API key needed."""
    url = "https://api.exchangerate-api.com/v4/latest/USD"
    response = httpx.get(url, timeout=5.0)
    response.raise_for_status()   # Raises error if 4xx or 5xx response
    data = response.json()         # JSON string → Python dict
    return data["rates"]["INR"]

# The async version is identical but with async/await
async def fetch_usd_inr_async():
    async with httpx.AsyncClient(timeout=5.0) as client:
        r = await client.get("https://api.exchangerate-api.com/v4/latest/USD")
        r.raise_for_status()
        return r.json()["rates"]["INR"]

try:
    rate = fetch_usd_inr_sync()
    print(f"USD/INR: {rate}")
except Exception as e:
    print(f"API call failed (no internet in this env): {type(e).__name__}")
    print("Fallback: 83.5")

In [ ]:
# ── asyncio.gather — running multiple API calls at once ───────
# Bug 6 note: asyncio should be imported at MODULE level, not inside functions.

import asyncio  # ← correct placement: top of file

async def fake_api_call(name, seconds):
    await asyncio.sleep(seconds)
    return f"{name} result"

async def fetch_all_markets():
    # gather runs all three at the same time
    # Total time = longest single call, NOT sum of all calls
    nse, forex, gold = await asyncio.gather(
        fake_api_call("NSE", 1.5),
        fake_api_call("Forex", 0.8),
        fake_api_call("Gold", 1.2),
    )
    return nse, forex, gold

import time
start = time.time()
results = await fetch_all_markets()
elapsed = time.time() - start

print(f"Results: {results}")
print(f"Time:    {elapsed:.1f}s  (would be 3.5s sequential, is {elapsed:.1f}s concurrent)")

In [ ]:
# ── Try/except with fallbacks — critical for demo reliability ─
# NEVER let a failed API call crash your app during a demo.
# Always have fallback values.

FALLBACKS = {
    "usd_inr": 83.5,
    "nifty_pe": 22.3,
    "repo_rate": 6.5,
    "cpi_inflation": 5.10,   # Stored as percent (5.1), not decimal (0.051)
}

async def safe_fetch_usd_inr():
    try:
        async with httpx.AsyncClient(timeout=5.0) as client:
            r = await client.get("https://api.exchangerate-api.com/v4/latest/USD")
            r.raise_for_status()
            return float(r.json()["rates"]["INR"])
    except Exception as e:
        print(f"API failed: {e}. Using fallback.")
        return FALLBACKS["usd_inr"]   # App continues working!

rate = await safe_fetch_usd_inr()
print(f"USD/INR rate: {rate}")

# ── Why cpi_inflation is stored as 5.1 not 0.051 ─────────────
# The market snapshot dict stores inflation in human-readable % form (5.1)
# so it can be displayed directly in the UI/prompt ("CPI: 5.1%")
# This is why Bug 3 (the ternary) needs to convert: raw / 100 → 0.051
# The fix is to always read once and convert once, never three times.

raw = FALLBACKS["cpi_inflation"]
decimal_inflation = raw / 100 if raw > 1 else raw
print(f"Stored as: {raw}%  →  Used in formulas as: {decimal_inflation}")

### 📚 Resources to learn HTTP and APIs in Python

- **httpx docs** — `python-httpx.org` — Official docs. Read 'Quickstart' and 'Async' sections.
- **Real Python — requests tutorial** — Search 'Python requests library Real Python' — httpx works almost identically.
- **NSE India API community** — Search 'NSE India API Python GitHub' — NSE has no official API docs, but the community has reverse-engineered it. Look at the `jugaad-trader` and `nsetools` GitHub repos for working examples.
- **Alpha Vantage docs** — `alphavantage.co/documentation` — Clear docs for stock/forex/commodity data. Free tier: 25 calls/day.
- **ExchangeRate-API docs** — `exchangerate-api.com/docs` — No key needed for basic rates. 1,500 free requests/month.
- **asyncio tutorial** — Search 'Python asyncio tutorial Corey Schafer YouTube' — the best visual explanation of async Python.
- **return_exceptions in asyncio.gather** — `docs.python.org/3/library/asyncio-task.html` — When set to True, exceptions from individual coroutines are returned as results, not raised. Market data already uses this correctly.

---
## Part 4 — Behaviour Profiler (`behaviour_profiler.py`)

This is the USP. It takes quiz answers and financial data and produces a persona that makes the AI's tone and advice deeply personal.

In [ ]:
# ── How persona scoring works ──────────────────────────────────
# A simple points system — each quiz answer adds points
# to one or more persona buckets. Highest bucket wins.

def simple_persona_demo(quiz_answers, savings_rate, experience_pct):
    scores = {"fire": 0, "yolo": 0, "family": 0, "debt": 0, "wealth": 0}

    q1_scores = {
        "invest_all":           {"fire": 3, "wealth": 2},
        "invest_half_travel":   {"yolo": 3},
        "pay_off_debt":         {"debt": 4},
        "buy_something_wanted": {"yolo": 2},
        "save_for_family_goal": {"family": 4},
    }
    for persona, points in q1_scores.get(quiz_answers[0], {}).items():
        scores[persona] += points

    if savings_rate >= 0.40:  scores["fire"] += 2
    if experience_pct >= 35:  scores["yolo"] += 2

    winner = max(scores, key=scores.get)
    return {"persona": winner, "scores": scores}

# Test cases
test_cases = [
    {"answers": ["invest_all", "not_having_enough_to_retire", "before_45"],
     "savings": 0.45, "exp": 10, "expected": "fire"},
    {"answers": ["invest_half_travel", "missing_out_on_experiences", "flexible_whenever_i_can"],
     "savings": 0.15, "exp": 45, "expected": "yolo"},
    {"answers": ["save_for_family_goal", "being_a_burden_on_family", "at_60"],
     "savings": 0.25, "exp": 15, "expected": "family"},
    {"answers": ["pay_off_debt", "stuck_in_debt_forever", "flexible_whenever_i_can"],
     "savings": 0.05, "exp": 10, "expected": "debt"},
]

for tc in test_cases:
    result = simple_persona_demo(tc["answers"], tc["savings"], tc["exp"])
    status = "✅" if result["persona"] == tc["expected"] else "❌"
    print(f"{status} Expected: {tc['expected']:7s} | Got: {result['persona']:7s} | Scores: {result['scores']}")

In [ ]:
# ── Bug 8 Demo: Balanced Builder missing + tie-breaking ───────

import random

scores_tied = {"fire": 4, "yolo": 4, "family": 2, "debt": 1, "wealth": 4}

# Python's max() on a dict returns the FIRST key with the max value
# In a tie: it depends on insertion order (Python 3.7+ guarantees dict order)
winner = max(scores_tied, key=scores_tied.get)
print(f"Tie between fire/yolo/wealth — max() returns: '{winner}'")
print(f"This is deterministic but NOT meaningful — all three tied at 4\n")

# ✅ Better approach: detect near-ties and use 'balanced'
def detect_persona_with_balanced(scores):
    sorted_vals = sorted(scores.values(), reverse=True)
    top_score  = sorted_vals[0]
    second_score = sorted_vals[1]

    if top_score - second_score <= 2:   # Gap of 2 or less = too close to call
        return "balanced"
    return max(scores, key=scores.get)

print(f"With balanced detection: '{detect_persona_with_balanced(scores_tied)}'")

clear_winner = {"fire": 10, "yolo": 3, "family": 2, "debt": 1, "wealth": 5}
print(f"Clear winner case: '{detect_persona_with_balanced(clear_winner)}'")

In [ ]:
# ── dataclass — a clean way to define structured data ─────────
from dataclasses import dataclass, asdict

@dataclass
class Persona:
    type: str
    tone: str
    risk_score: int
    primary_motivation: str
    emoji_label: str

fire_persona = Persona(
    type="FIRE Seeker",
    tone="structured",
    risk_score=8,
    primary_motivation="Freedom from the 9-to-5",
    emoji_label="Target: Early retirement"
)

print("Persona object:", fire_persona)
print("\nAs dict (for JSON):", asdict(fire_persona))

# Why dataclass instead of a plain dict?
# 1. Autocomplete in your IDE — fire_persona.tone not fire_persona["tone"]
# 2. Type checking — if you pass the wrong type, tools warn you
# 3. asdict() gives you JSON-serializable output for free

### 📚 Resources to understand behaviour profiling

- **Python dataclasses** — Search 'Python dataclasses tutorial Real Python' — 15-minute read.
- **ET Money persona quiz** — Visit `etmoney.com` and take their risk assessment quiz. Study how they ask questions and map answers to risk profiles — direct competitor research.
- **SEBI risk profiling questionnaire** — Search 'SEBI risk profiling questionnaire 2023' — the official template all SEBI-registered advisors must use. Your quiz should align with this.
- **IRDAI Consumer Education** — `irdai.gov.in/consumer-education` — Insurance need estimation guidelines.

---
## Part 5 — System Prompt Engineering (`system_prompt.py`)

The system prompt is the single most important thing you write. It's the instructions you give the AI. A bad prompt = generic, useless responses. A great prompt = responses that feel like a real financial advisor.

In [ ]:
# ── What makes a great system prompt? ────────────────────────

bad_prompt = "You are a financial advisor. Help the user."

good_prompt_template = """
You are FinMentor — an AI personal finance advisor for India.

USER CONTEXT:
- Age: 26, City: Mumbai, Savings rate: 28%
- Monthly income: ₹75,000, Monthly expenses: ₹54,000
- FIRE corpus needed: ₹1.62 crore, FIRE age: 48
- Emergency fund: Covered only 2.1 months (shortfall: ₹2.1 lakh)
- Term insurance gap: ₹54 lakh

LIVE MARKET:
- Nifty P/E: 22.3 (fair value) — use 50/35/10/5 equity/debt/gold/liquid split
- Repo rate: 6.5% — FDs giving 7.25%
- CPI inflation: 5.1%

PERSONA: FIRE Seeker — tone: structured, number-heavy.
Show FIRE date impact for every suggestion. Be precise.

RULES:
1. Emergency fund first, insurance second, investments third.
2. Use ₹ and lakh/crore notation.
3. End investment advice with SEBI disclaimer.
"""

print(f"Bad prompt:  {len(bad_prompt)} characters — vague, gives AI nothing to work with")
print(f"Good prompt: {len(good_prompt_template)} characters — specific numbers, persona, rules")
print("\nLonger, specific prompts almost always produce better output.")

In [ ]:
# ── f-strings — how system_prompt.py injects data ─────────────

def inr(amount):
    """Format a number as Indian currency string."""
    if amount >= 10_000_000:
        return f"₹{amount/10_000_000:.2f} crore"
    elif amount >= 100_000:
        return f"₹{amount/100_000:.2f} lakh"
    return f"₹{amount:,.0f}"

# Test the formatter
test_amounts = [45000, 250000, 1_620_000, 15_000_000]
for amt in test_amounts:
    print(f"  {amt:>12,}  →  {inr(amt)}")

In [ ]:
# ── Persona-driven prompt variations ─────────────────────────
# The framing field in each Persona changes HOW the AI responds
# to the exact same financial situation.

SITUATION = {
    "name": "Riya", "age": 29, "income": 90000, "expense": 60000,
    "sip_needed": 8500
}

PERSONA_FRAMINGS = {
    "fire": (
        "Always frame advice as how many months closer to FIRE. "
        "Be number-precise. No lifestyle advice."
    ),
    "yolo": (
        "NEVER suggest cutting travel or experiences. Show how automating "
        "SIPs first leaves the lifestyle 100% intact."
    ),
    "family": (
        "Lead with family protection: insurance → education fund → retirement. "
        "Be warm, acknowledge the emotional weight."
    ),
}

for persona_type, framing in PERSONA_FRAMINGS.items():
    print(f"\n{'─'*55}")
    print(f"PERSONA: {persona_type.upper()}")
    print(f"AI instruction: {framing}")
    print(f"The same ₹{SITUATION['sip_needed']:,}/month recommendation will be")
    print(f"framed completely differently for each persona.")

### 📚 Resources to learn prompt engineering

- **Anthropic Prompt Engineering Guide** — `docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/overview` — The most comprehensive, free guide. Principles apply universally across Gemini, GPT, and Claude.
- **Google AI Studio** — `aistudio.google.com` — Test your system prompts directly here before coding. Fastest way to iterate without running a server.
- **Lilian Weng's prompt engineering post** — Search 'Lilian Weng prompt engineering' — Deep technical breakdown. Long but excellent.
- **Learn Prompting** — `learnprompting.org` — Free open-source course entirely on prompt engineering. Beginner-friendly.
- **Structured prompts with XML tags** — Gemini and Claude both respond better when you use clear delimiters like `━━━` or `<user_context>` tags rather than plain paragraphs.

---
## Part 6 — Gemini API Integration

How to actually call Gemini and get a response.

In [ ]:
# ── Basic Gemini call ─────────────────────────────────────────
# pip install google-generativeai

import google.generativeai as genai
import os

# Get your free API key at: aistudio.google.com
# genai.configure(api_key="YOUR_KEY_HERE")

def ask_gemini_simple(system_prompt, user_message):
    model = genai.GenerativeModel(
        model_name="gemini-2.0-flash",      # Fast and free tier available
        system_instruction=system_prompt,
    )
    response = model.generate_content(user_message)
    return response.text

# Example (uncomment to test — needs API key):
# reply = ask_gemini_simple(
#     system_prompt="You are a friendly Indian finance advisor.",
#     user_message="Should I invest in ELSS or PPF?"
# )
# print(reply)

print("Gemini API structure ready. Add your API key to test it.")

In [ ]:
# ── Multi-turn conversation (chat history) ────────────────────
# For a real chat app, the AI needs to remember previous messages.
# Gemini does this via start_chat() with a history list.
# CRITICAL: Gemini has NO memory between API calls.
# You must send the FULL history on every single request.

def multi_turn_demo():
    # History format: list of {role, parts} dicts
    # role must be exactly "user" or "model" (not "assistant")
    history = [
        {"role": "user",  "parts": ["My income is ₹80,000/month"]},
        {"role": "model", "parts": ["Got it! What are your monthly expenses?"]},
        {"role": "user",  "parts": ["Expenses are ₹55,000/month"]},
        {"role": "model", "parts": ["Your savings rate is 31% — that's healthy."]},
    ]

    print("Chat history structure:")
    for i, turn in enumerate(history):
        print(f"  Turn {i+1} [{turn['role']:5s}]: {turn['parts'][0][:60]}")

    print("\nKey rules:")
    print("  1. role must be 'user' or 'model' (NOT 'assistant')")
    print("  2. history must alternate user/model — no two consecutive same roles")
    print("  3. Send FULL history every time — Gemini has zero memory between calls")
    print("  4. Store history in React state (useState), pass it in every /chat request")

multi_turn_demo()

In [ ]:
# ── Environment variables — never hardcode API keys ───────────
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env file in project root

api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    print(f"API key loaded (length: {len(api_key)} chars)")
else:
    print("GEMINI_API_KEY not set — create a .env file from .env.example")

# .env file format (never commit this file to GitHub!):
# GEMINI_API_KEY=AIzaSy...
# ALPHA_VANTAGE_KEY=ABCD1234...

# .gitignore must include:
# .env
# __pycache__/
# venv/
print("\n.gitignore must contain: .env")
print("If you accidentally commit your key, revoke it immediately at aistudio.google.com")

In [ ]:
# ── Error handling for Gemini API calls ───────────────────────
# Gemini can fail for: rate limits, content safety, network issues.
# The current main.py raises HTTPException 500 on any failure.
# Better: distinguish between error types.

from fastapi import HTTPException
import google.generativeai as genai

async def safe_gemini_call(model, message, history):
    """
    Improved Gemini caller with better error classification.
    """
    try:
        if history:
            session = model.start_chat(history=history)
            response = session.send_message(message)
        else:
            response = model.generate_content(message)

        if not response.text:
            # Safety filter blocked the response
            return "I'm not able to respond to that. Please rephrase your financial question."

        return response.text

    except Exception as e:
        error_str = str(e).lower()
        if "quota" in error_str or "rate" in error_str:
            raise HTTPException(status_code=429, detail="API rate limit reached. Please try again in a moment.")
        elif "safety" in error_str or "blocked" in error_str:
            raise HTTPException(status_code=400, detail="Request blocked by safety filter.")
        else:
            raise HTTPException(status_code=500, detail=f"AI generation failed: {str(e)}")

print("Better error handling: classify quota/safety/network errors separately.")
print("Status 429 → frontend shows 'please wait'")
print("Status 400 → frontend shows 'rephrase your question'")
print("Status 500 → frontend shows 'server error'")

### 📚 Resources to learn Gemini API and LLM integration

- **Google AI Studio** — `aistudio.google.com` — Get your free API key, test prompts, see Python code examples. Start here.
- **Gemini API Python quickstart** — `ai.google.dev/gemini-api/docs/quickstart?lang=python` — Official 10-minute guide. Copy-paste ready.
- **Gemini model names** — `ai.google.dev/gemini-api/docs/models/gemini` — Model list with context windows and pricing. `gemini-2.0-flash` is the best free-tier choice.
- **python-dotenv docs** — `pypi.org/project/python-dotenv` — Simple but critical for API key management.
- **FastAPI HTTPException docs** — `fastapi.tiangolo.com/tutorial/handling-errors/` — Proper HTTP error codes for each failure scenario.

---
## Part 7 — Knowledge Base (`knowledge_base.py`)

This is the financial rules engine — all domain knowledge encoded as Python dicts.

In [ ]:
# ── How the knowledge base is structured ─────────────────────
# Just a Python dict — but a very carefully curated one.
# Every value comes from a credible source (SEBI, AMFI, RBI, Income Tax Act).

KB_DEMO = {
    "deductions": {
        "80C_limit": 150000,        # Income Tax Act Section 80C (FY25)
        "80CCD_1B_nps": 50000,      # Additional NPS deduction
        "80D_self_below60": 25000,  # Health insurance premium
    },
    "rules_of_thumb": {
        "emergency_fund_months": 6,
        "term_insurance_multiplier": 12,
        "fire_corpus_multiplier": 25,
    },
    "historical_returns": {
        "nifty50_20yr_cagr": 0.126,  # NSE historical data (TRI basis)
        "ppf_rate": 0.071,            # Government declared quarterly
        "gold_20yr_cagr_inr": 0.11,  # Historical INR gold returns
    }
}

print(f"80C Limit:      ₹{KB_DEMO['deductions']['80C_limit']:,}")
print(f"FIRE multiplier: {KB_DEMO['rules_of_thumb']['fire_corpus_multiplier']}x annual expenses")
print(f"Nifty 20yr CAGR: {KB_DEMO['historical_returns']['nifty50_20yr_cagr']*100:.1f}%")

In [ ]:
# ── Why hardcode these instead of asking the AI? ─────────────
# LLMs sometimes hallucinate numbers.
# A wrong tax limit or insurance rule could give someone harmful financial advice.
# By hardcoding from verified sources, you guarantee accuracy.

sources = {
    "Tax slabs": "incometax.gov.in → 'Tax Rates' section",
    "PPF rate": "dea.gov.in → 'Small Savings' (updated quarterly by government)",
    "Nifty CAGR": "nseindia.com → Historical Data → Nifty 50 TRI",
    "SEBI rules": "sebi.gov.in → Circulars",
    "Insurance norms": "irdai.gov.in → Consumer Information",
    "NPS limits": "npstrust.org.in → Contribution & Withdrawal",
}

print("Where to verify each value in knowledge_base.py:")
for rule, source in sources.items():
    print(f"  {rule:20s}: {source}")

print("\n⚠️  Update knowledge_base.py at the start of each financial year (April).")
print("    Tax slabs, PPF rate, and standard deduction change most often.")

In [ ]:
# ── Asset allocation with market tilt — how it actually works ─
# calc_asset_allocation() reads the base allocation from KB
# and then shifts equity/debt based on Nifty P/E.

KB_ALLOC = {
    "conservative":    {"equity": 0.20, "debt": 0.60, "gold": 0.10, "liquid": 0.10},
    "moderate":        {"equity": 0.50, "debt": 0.35, "gold": 0.10, "liquid": 0.05},
    "aggressive":      {"equity": 0.75, "debt": 0.15, "gold": 0.05, "liquid": 0.05},
    "very_aggressive": {"equity": 0.85, "debt": 0.10, "gold": 0.00, "liquid": 0.05},
}

def show_allocation(profile, nifty_pe):
    base = dict(KB_ALLOC[profile])
    note = ""
    if nifty_pe > 24:
        base["equity"] = max(0.10, base["equity"] - 0.10)
        base["debt"]   = min(0.80, base["debt"]   + 0.10)
        note = f"P/E {nifty_pe} > 24 → shifted 10% from equity to debt"
    elif nifty_pe < 18:
        base["equity"] = min(0.90, base["equity"] + 0.10)
        base["debt"]   = max(0.05, base["debt"]   - 0.10)
        note = f"P/E {nifty_pe} < 18 → shifted 10% from debt to equity"
    else:
        note = f"P/E {nifty_pe} in fair value range (18-24) → base allocation"

    print(f"\n{profile.upper()} at P/E {nifty_pe}:")
    print(f"  Equity {base['equity']*100:.0f}% | Debt {base['debt']*100:.0f}% | Gold {base['gold']*100:.0f}% | Liquid {base['liquid']*100:.0f}%")
    print(f"  Note: {note}")

show_allocation("moderate", nifty_pe=22.3)  # Fair value — base allocation
show_allocation("moderate", nifty_pe=27.5)  # Overvalued — tilt to debt
show_allocation("moderate", nifty_pe=15.0)  # Cheap — tilt to equity
show_allocation("aggressive", nifty_pe=22.3)

---
## Part 8 — Running Everything Together

In [ ]:
# ── Full pipeline simulation — no server needed ────────────────
# Simulates one complete /chat request end-to-end.

sample_user = {
    "name": "Arjun",
    "age": 27,
    "city": "Pune",
    "city_tier": "metro",
    "dependants": 0,
    "tax_regime": "new",
    "monthly_income": 80000,
    "monthly_expenses": 52000,
    "existing_investments": 300000,
    "outstanding_debt": 500000,
    "liquid_savings": 80000,
    "term_insurance_cover": 5000000,
    "health_insurance_cover": 500000,
    "existing_80c_investments": 72000,
    "existing_nps_contribution": 0,
    "health_premium_paid": 12000,
    "goals": ["retirement", "buy_home"],
}

sample_quiz = ["invest_all", "not_having_enough_to_retire", "before_45"]

# ── Step 1: Savings rate ──────────────────────────────────────
savings_rate = (sample_user["monthly_income"] - sample_user["monthly_expenses"]) / sample_user["monthly_income"]
print(f"[Step 1] Savings rate: {savings_rate*100:.1f}%")

# ── Step 2: FIRE corpus (with fixed inflation handling) ───────
raw_inflation = 5.1  # as stored in market snapshot (percentage)
inflation = raw_inflation / 100 if raw_inflation > 1 else raw_inflation  # Bug 3 fix
fire_corpus = sample_user["monthly_expenses"] * 12 * (1 + inflation) * 25
print(f"[Step 2] FIRE corpus target: ₹{fire_corpus/100_000:.1f} lakh")

# ── Step 3: SIP needed for retirement by 60 ───────────────────
years = 60 - sample_user["age"]
r = 0.12 / 12
n = years * 12
sip = fire_corpus * r / ((1 + r)**n - 1)
print(f"[Step 3] Monthly SIP for retirement by 60: ₹{sip:,.0f}")

# ── Step 4: Emergency fund check ──────────────────────────────
emergency_target = sample_user["monthly_expenses"] * 6
emergency_gap = max(0, emergency_target - sample_user["liquid_savings"])
months_covered = sample_user["liquid_savings"] / sample_user["monthly_expenses"]
print(f"[Step 4] Emergency fund: {months_covered:.1f} months covered (need 6) | Gap: ₹{emergency_gap/1000:.0f}K")

# ── Step 5: Insurance gap ─────────────────────────────────────
insurance_target = sample_user["monthly_income"] * 12 * 12
insurance_gap = max(0, insurance_target - sample_user["term_insurance_cover"])
print(f"[Step 5] Term insurance gap: ₹{insurance_gap/100_000:.0f}L")

# ── Step 6: Persona detection ─────────────────────────────────
# invest_all → fire+3, wealth+2
# not_having_enough_to_retire → fire+3
# before_45 → fire+4
# savings_rate=0.35 → fire+2
scores = {"fire": 3+3+4+2, "yolo": 0, "family": 0, "debt": 0, "wealth": 2}
print(f"[Step 6] Persona scores: {scores}")
print(f"         Persona: FIRE Seeker (score: {scores['fire']})")

print("\n✅ All steps working correctly. Connect to Gemini to get the final response.")

In [ ]:
# ── How to start the server ───────────────────────────────────
startup_commands = """
# 1. Navigate to your project folder
cd your-project/server

# 2. Create virtual environment (run once)
python -m venv venv
source venv/bin/activate        # Mac/Linux
.\\venv\\Scripts\\activate       # Windows

# 3. Install dependencies
pip install -r requirements.txt

# 4. Set up environment (copy template, add your Gemini key)
cp .env.example .env
# Edit .env → GEMINI_API_KEY=AIzaSy...

# 5. Run the server
uvicorn main:app --reload --port 8000

# 6. View auto-generated API docs
open http://localhost:8000/docs

# 7. Test health check
curl http://localhost:8000/health
"""
print(startup_commands)

---
## Part 9 — Testing Your API

Once the server is running, use these test payloads to verify every endpoint works correctly.

In [ ]:
# ── Sample /chat request payload ──────────────────────────────
# Copy this into Postman or FastAPI /docs to test the endpoint.

import json

sample_chat_payload = {
    "user": {
        "name": "Arjun",
        "age": 27,
        "city": "Pune",
        "city_tier": "metro",
        "dependants": 0,
        "tax_regime": "new",
        "monthly_income": 80000,
        "monthly_expenses": 52000,
        "existing_investments": 300000,
        "outstanding_debt": 500000,
        "liquid_savings": 80000,
        "term_insurance_cover": 5000000,
        "health_insurance_cover": 500000,
        "existing_80c_investments": 72000,
        "existing_nps_contribution": 0,
        "health_premium_paid": 12000,
        "goals": ["retirement", "buy_home"]
    },
    "quiz_answers": [
        "invest_all",
        "not_having_enough_to_retire",
        "before_45"
    ],
    "experience_spend_pct": 15,
    "message": "Where should I start? I feel lost about money.",
    "conversation_history": []
}

print("POST /chat payload:")
print(json.dumps(sample_chat_payload, indent=2))

In [ ]:
# ── Testing edge cases — important before demo day ────────────
# These are the inputs most likely to break your app.
# Test each one with Postman before the hackathon presentation.

edge_cases = [
    {
        "name": "Expenses > Income (in debt)",
        "payload": {"monthly_income": 40000, "monthly_expenses": 55000},
        "expect": "Negative savings rate — should not crash. FIRE not achievable."
    },
    {
        "name": "Age = 18 (youngest allowed)",
        "payload": {"age": 18, "monthly_income": 25000, "monthly_expenses": 20000},
        "expect": "FIRE date 42 years away — very long horizon, should still compute."
    },
    {
        "name": "Age = 70 (oldest allowed)",
        "payload": {"age": 70, "monthly_income": 50000, "monthly_expenses": 40000},
        "expect": "years_to_60 = max(1, -10) = 1 — SIP horizon is 1 year, very high SIP."
    },
    {
        "name": "Invalid quiz answer (Bug 7)",
        "payload": {"quiz_answers": ["INVALID_KEY", "missing_out_on_experiences", "before_45"]},
        "expect": "Before bug fix: wrong persona silently. After fix: 422 validation error."
    },
    {
        "name": "Wrong quiz_answers count (Bug 1)",
        "payload": {"quiz_answers": ["invest_all"]},  # Only 1 answer
        "expect": "Before bug fix: accepted silently. After fix: 422 Unprocessable Entity."
    },
    {
        "name": "High savings rate (FIRE track)",
        "payload": {"monthly_income": 200000, "monthly_expenses": 60000},
        "expect": "Savings rate 70% — FIRE Seeker persona very likely, early FIRE date."
    },
]

print("Edge cases to test before your hackathon demo:")
print()
for i, case in enumerate(edge_cases, 1):
    print(f"{i}. {case['name']}")
    print(f"   Expected: {case['expect']}")
    print()

---
## Part 10 — Complete Corrected Code (Copy-Paste Ready)

All 8 bugs fixed in one place.

In [ ]:
# ── CORRECTED main.py snippets (apply these patches) ─────────

# ─── PATCH 1: Fix imports (Bugs 1, 2) ─────────────────────────
corrected_import = """
from pydantic import BaseModel, Field, field_validator   # ← field_validator, not validator
"""

# ─── PATCH 2: Fix ChatRequest (Bug 1) ─────────────────────────
corrected_chat_request = """
class ChatRequest(BaseModel):
    user: UserProfile
    quiz_answers: list[str] = Field(
        ...,
        min_length=3,      # ← was min_items (Pydantic v1, silently ignored)
        max_length=3,      # ← was max_items (Pydantic v1, silently ignored)
        description="Exactly 3 quiz answer keys from onboarding"
    )
    experience_spend_pct: Optional[int] = Field(20, ge=0, le=100)
    message: str = Field(..., min_length=1, max_length=2000)
    conversation_history: Optional[list[dict]] = []
"""

# ─── PATCH 3: Fix validator (Bug 2) ───────────────────────────
corrected_validator = """
class UserProfile(BaseModel):
    # ... other fields ...

    @field_validator("monthly_expenses")          # ← @field_validator, not @validator
    @classmethod                                   # ← required in Pydantic v2
    def expenses_less_than_income(cls, v, info):  # ← info, not values
        income = info.data.get("monthly_income")  # ← info.data, not values["income"]
        if income and v > income:
            pass  # warn but don't block
        return v
"""

# ─── PATCH 4: Fix inflation ternary (Bug 3) ───────────────────
corrected_inflation = """
def run_calculations(user: UserProfile, nifty_pe: float, market: dict) -> dict:
    # ✅ Fix Bug 3: read once, convert once
    raw_inflation = market.get("cpi_inflation", 6.0)   # always stored as percent
    inflation = raw_inflation / 100 if raw_inflation > 1 else raw_inflation

    fire = calc_fire_date(
        current_age=user.age,
        monthly_income=user.monthly_income,
        monthly_expense=user.monthly_expenses,
        current_corpus=user.existing_investments,
        annual_return=0.12,
        inflation=inflation,    # ← clean, always correct
    )
    ...
"""

# ─── PATCH 5: Fix user.dict() (Bug 5) ─────────────────────────
corrected_dict = """
system = build_system_prompt(
    user=user.model_dump(),    # ← .model_dump(), not .dict()
    market=market,
    ...
)
"""

print("All patches printed above. Apply them to main.py.")
print("Bug 4 (calc_fire_date) and Bug 6 (asyncio import) are fixed in financial_engine.py")
print("and market_data.py respectively — see earlier cells for the full corrected functions.")

In [ ]:
# ── CORRECTED financial_engine.py: calc_fire_date (Bug 4) ─────

def calc_fire_date_corrected(
    current_age: int,
    monthly_income: float,
    monthly_expense: float,
    current_corpus: float,
    annual_return: float = 0.12,
    inflation: float = 0.06,
    salary_growth: float = 0.08,    # NEW param — typical Indian salary increment
) -> dict:
    """
    Corrected FIRE date estimator.
    Key fix: monthly_savings recalculated each year as both
    income (grows with salary) and expenses (grow with inflation) change.
    """
    fire_corpus_target = monthly_expense * 12 * (1 + inflation) * 25
    monthly_savings = monthly_income - monthly_expense
    corpus = current_corpus
    age = current_age

    for year in range(1, 51):
        corpus = corpus * (1 + annual_return)        # Grow existing corpus
        corpus += monthly_savings * 12               # Add this year's savings

        # ✅ Update BOTH income and expense, then recalculate savings
        monthly_income  *= (1 + salary_growth)       # Salary grows each year
        monthly_expense *= (1 + inflation)            # Expenses grow each year
        monthly_savings  = monthly_income - monthly_expense  # Recalculate!

        fire_corpus_target = monthly_expense * 12 * (1 + inflation) * 25
        age += 1

        if corpus >= fire_corpus_target:
            return {
                "fire_age": age,
                "years_to_fire": year,
                "fire_corpus_target": round(fire_corpus_target, 2),
                "projected_corpus": round(corpus, 2),
                "achievable": True,
            }

    return {
        "fire_age": None,
        "years_to_fire": None,
        "fire_corpus_target": round(fire_corpus_target, 2),
        "projected_corpus": round(corpus, 2),
        "achievable": False,
        "note": "FIRE not achievable in 50 years. Increase savings rate or reduce expense growth.",
    }

# Test with Arjun's profile
result = calc_fire_date_corrected(
    current_age=27,
    monthly_income=80000,
    monthly_expense=52000,
    current_corpus=300000,
)
print("Corrected FIRE calculation for Arjun:")
for k, v in result.items():
    print(f"  {k}: {v}")

---
## Complete Learning Roadmap

Prioritised list of what to learn, in what order, with exact resources.

### Week 1 — Backend foundations

| Topic | Resource | Time |
|-------|----------|------|
| FastAPI basics | `fastapi.tiangolo.com` — 'First Steps' to 'Request Body' | 3 hrs |
| Pydantic v2 (critical — project uses v2) | `docs.pydantic.dev/latest/migration/` — migration guide | 2 hrs |
| async/await Python | Corey Schafer 'Python asyncio' on YouTube | 1.5 hrs |
| httpx for API calls | `python-httpx.org` — 'Quickstart' + 'Async' | 1 hr |
| python-dotenv | `pypi.org/project/python-dotenv` | 20 min |

### Week 1 — Finance knowledge (run in parallel)

| Topic | Resource | Time |
|-------|----------|------|
| SIP & compound interest | `amfiindia.com/investor-education` — SIP section | 1 hr |
| FIRE concept for India | `freefincal.com` — search 'FIRE India' | 1.5 hrs |
| Indian tax saving (80C, NPS, 80D) | `cleartax.in/s/income-tax-deductions` | 2 hrs |
| Asset allocation | `etmoney.com/learn/mutual-funds/asset-allocation` | 1 hr |
| Insurance basics | `policybazaar.com/life-insurance/term-insurance/articles` | 1 hr |
| Must-read book | Monika Halan — *Let's Talk Money* (chapters 3, 5, 7) | 3 hrs |

### Week 2 — AI/LLM integration

| Topic | Resource | Time |
|-------|----------|------|
| Gemini API | `ai.google.dev/gemini-api/docs/quickstart?lang=python` | 1 hr |
| Prompt engineering | `learnprompting.org` — Beginner track | 3 hrs |
| Anthropic prompt guide (universal) | `docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/overview` | 2 hrs |
| Google AI Studio | `aistudio.google.com` — experiment with prompts live | Ongoing |

### Developer tools

- **Postman** (`postman.com`) — Test every endpoint before connecting the frontend. Essential.
- **FastAPI /docs** — `localhost:8000/docs` — Auto-generated interactive API explorer. Test all endpoints here.
- **Thunder Client** — VS Code extension, same as Postman but inside your editor.
- **Google AI Studio** — Test system prompts without running a server.

---

## Quick Reference: File responsibilities

```
server/
├── main.py               → HTTP routes, request validation, orchestration
├── financial_engine.py   → Pure math: SIP, FIRE, compound interest, insurance gap
├── market_data.py        → Fetches NSE/RBI/Forex live data with fallbacks
├── behaviour_profiler.py → Quiz answers + financial data → persona type
├── system_prompt.py      → Assembles all data into the AI instruction string
├── knowledge_base.py     → Static rules: tax limits, SEBI norms, benchmarks
├── requirements.txt      → Python packages to install
└── .env.example          → Template for API keys
```

**When something goes wrong — which file to edit:**

| Problem | File to edit |
|---------|-------------|
| AI response quality is bad | `system_prompt.py` |
| Wrong FIRE date / SIP numbers | `financial_engine.py` |
| Market data stale / wrong | `market_data.py` (update FALLBACKS) |
| Wrong persona detected | `behaviour_profiler.py` (adjust scores) |
| Tax limits changed (new financial year) | `knowledge_base.py` |
| New API endpoint needed | `main.py` |
| React app getting CORS errors | `main.py` (allow_origins list) |